# Levi's India - RFM Analysis & Customer Segmentation

**Portfolio Project**: Complete end-to-end customer analytics

---

## 📋 Project Overview

This notebook analyzes **50,000 Levi's India customers** using **RFM (Recency, Frequency, Monetary)** methodology to:
- Segment customers into behavioral groups
- Identify high-value segments and at-risk customers
- Provide data-driven marketing campaign recommendations

**Expected Business Impact**: 
- Protect ₹22 Cr Champions segment
- Recover ₹0.7 Cr from at-risk VIPs
- Convert 3,739 new customers (3-4x LTV potential)

---

## 🎯 What is RFM?

**RFM = Recency, Frequency, Monetary**

- **Recency (R)**: How recently did they purchase? (Days since last order)
- **Frequency (F)**: How often do they purchase? (Number of transactions)
- **Monetary (M)**: How much do they spend? (Total revenue)

Customers are scored 1-5 on each dimension and grouped into segments like Champions, Loyal, At-Risk, Lost.

---

## Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("✅ Libraries imported successfully")

---

## Step 2: Generate Synthetic Dataset

We'll create realistic customer and transaction data that mirrors real-world patterns.

In [ ]:
# Set seed for reproducibility
np.random.seed(42)

print("🏭 Generating Levi's India Customer Dataset...")
print("=" * 70)

# Configuration
NUM_CUSTOMERS = 50000
START_DATE = datetime(2020, 1, 1)
END_DATE = datetime(2022, 12, 31)
TOTAL_DAYS = (END_DATE - START_DATE).days

# Segment distribution (realistic proportions)
SEGMENT_DISTRIBUTION = {
    'Champions': 0.03,
    'Loyal': 0.06,
    'Potential_Loyal': 0.09,
    'New_Customers': 0.15,
    'Promising': 0.12,
    'Need_Attention': 0.10,
    'About_to_Sleep': 0.08,
    'At_Risk': 0.07,
    'Cant_Lose': 0.04,
    'Hibernating': 0.13,
    'Lost': 0.13
}

# Segment behavior parameters
SEGMENT_PARAMS = {
    'Champions': {
        'frequency': (6, 12),
        'recency_days': (0, 30),
        'avg_order_value': (5000, 9000),
        'product_variety': (3, 5),
        'discount_usage': 0.1,
    },
    'Loyal': {
        'frequency': (4, 6),
        'recency_days': (20, 60),
        'avg_order_value': (3500, 6000),
        'product_variety': (2, 4),
        'discount_usage': 0.2,
    },
    'Potential_Loyal': {
        'frequency': (3, 4),
        'recency_days': (40, 90),
        'avg_order_value': (3000, 5000),
        'product_variety': (2, 3),
        'discount_usage': 0.3,
    },
    'New_Customers': {
        'frequency': (1, 2),
        'recency_days': (0, 30),
        'avg_order_value': (2500, 4500),
        'product_variety': (1, 2),
        'discount_usage': 0.15,
    },
    'Promising': {
        'frequency': (3, 4),
        'recency_days': (50, 100),
        'avg_order_value': (3200, 5500),
        'product_variety': (2, 3),
        'discount_usage': 0.25,
    },
    'Need_Attention': {
        'frequency': (2, 3),
        'recency_days': (90, 150),
        'avg_order_value': (2800, 4500),
        'product_variety': (1, 2),
        'discount_usage': 0.4,
    },
    'About_to_Sleep': {
        'frequency': (2, 3),
        'recency_days': (120, 180),
        'avg_order_value': (2500, 4000),
        'product_variety': (1, 2),
        'discount_usage': 0.5,
    },
    'At_Risk': {
        'frequency': (3, 5),
        'recency_days': (180, 300),
        'avg_order_value': (4000, 7000),
        'product_variety': (2, 3),
        'discount_usage': 0.3,
    },
    'Cant_Lose': {
        'frequency': (5, 8),
        'recency_days': (200, 400),
        'avg_order_value': (5500, 10000),
        'product_variety': (3, 4),
        'discount_usage': 0.15,
    },
    'Hibernating': {
        'frequency': (1, 2),
        'recency_days': (300, 500),
        'avg_order_value': (2000, 3500),
        'product_variety': (1, 2),
        'discount_usage': 0.6,
    },
    'Lost': {
        'frequency': (1, 2),
        'recency_days': (500, 900),
        'avg_order_value': (1800, 3000),
        'product_variety': (1, 1),
        'discount_usage': 0.7,
    }
}

CHANNELS = ['Store', 'Online', 'Mobile_App']
CITIES = ['Mumbai', 'Delhi', 'Bangalore', 'Pune', 'Hyderabad', 'Chennai', 'Kolkata', 'Ahmedabad']
PRODUCT_CATALOG = {
    'Jeans': {'min': 2500, 'max': 8000, 'avg': 4500},
    'Shirts': {'min': 1500, 'max': 4500, 'avg': 2800},
    'T-Shirts': {'min': 800, 'max': 2500, 'avg': 1500},
    'Jackets': {'min': 4000, 'max': 12000, 'avg': 7500},
    'Accessories': {'min': 500, 'max': 2000, 'avg': 1200}
}

print(f"Configuration:")
print(f"  Customers: {NUM_CUSTOMERS:,}")
print(f"  Date Range: {START_DATE.date()} to {END_DATE.date()}")
print(f"  Segments: {len(SEGMENT_DISTRIBUTION)}")

In [ ]:
# Generate customers
print("\n📊 Generating customers...")

customers = []
customer_id = 1000

for segment, proportion in SEGMENT_DISTRIBUTION.items():
    num_in_segment = int(NUM_CUSTOMERS * proportion)
    params = SEGMENT_PARAMS[segment]
    
    for _ in range(num_in_segment):
        customers.append({
            'customer_id': f'CUST{customer_id:06d}',
            'segment': segment,
            'join_date': START_DATE + timedelta(days=np.random.randint(0, TOTAL_DAYS - 180)),
            'city': np.random.choice(CITIES),
            'preferred_channel': np.random.choice(CHANNELS),
            'frequency_target': np.random.randint(*params['frequency']),
            'recency_target': np.random.randint(*params['recency_days']),
            'aov_min': params['avg_order_value'][0],
            'aov_max': params['avg_order_value'][1],
            'product_variety': np.random.randint(*params['product_variety']),
            'discount_prone': np.random.random() < params['discount_usage']
        })
        customer_id += 1

customers_df = pd.DataFrame(customers)
print(f"✅ Generated {len(customers_df):,} customers")
print(f"\nSegment distribution:")
print(customers_df['segment'].value_counts())

In [ ]:
# Generate transactions
print("\n💳 Generating transactions...")

transactions = []
transaction_id = 100000

for idx, customer in customers_df.iterrows():
    num_orders = customer['frequency_target']
    days_active = (END_DATE - customer['join_date']).days
    
    if days_active <= 0 or num_orders == 0:
        continue
    
    last_order_date = END_DATE - timedelta(days=customer['recency_target'])
    
    # Generate order dates
    if num_orders == 1:
        order_dates = [last_order_date]
    else:
        time_span = (last_order_date - customer['join_date']).days
        if time_span > 0:
            intervals = sorted([np.random.randint(0, time_span) for _ in range(num_orders)])
            order_dates = [customer['join_date'] + timedelta(days=d) for d in intervals]
        else:
            order_dates = [last_order_date]
    
    # Generate each order
    for order_date in order_dates:
        num_items = np.random.choice([1, 2, 3, 4], p=[0.5, 0.3, 0.15, 0.05])
        order_value = 0
        order_items = []
        
        for _ in range(num_items):
            if customer['product_variety'] == 1:
                category = 'Jeans'
            else:
                category = np.random.choice(list(PRODUCT_CATALOG.keys()))
            
            base_price = np.random.uniform(
                PRODUCT_CATALOG[category]['min'],
                PRODUCT_CATALOG[category]['max']
            )
            
            if customer['discount_prone'] and np.random.random() < 0.6:
                discount_pct = np.random.choice([10, 15, 20, 25, 30])
                final_price = base_price * (1 - discount_pct / 100)
                has_discount = True
            else:
                final_price = base_price
                discount_pct = 0
                has_discount = False
            
            order_value += final_price
            order_items.append(category)
        
        order_value = np.clip(order_value, customer['aov_min'], customer['aov_max'])
        
        channel = customer['preferred_channel'] if np.random.random() < 0.8 else np.random.choice(CHANNELS)
        
        transactions.append({
            'transaction_id': f'TXN{transaction_id:08d}',
            'customer_id': customer['customer_id'],
            'order_date': order_date,
            'order_value': round(order_value, 2),
            'num_items': num_items,
            'channel': channel,
            'city': customer['city'],
            'has_discount': has_discount,
            'discount_pct': discount_pct if has_discount else 0,
            'product_categories': '|'.join(order_items)
        })
        transaction_id += 1

transactions_df = pd.DataFrame(transactions)
print(f"✅ Generated {len(transactions_df):,} transactions")
print(f"\n📈 Dataset Summary:")
print(f"  Total Revenue: ₹{transactions_df['order_value'].sum()/1e7:.2f} Crore")
print(f"  Avg Order Value: ₹{transactions_df['order_value'].mean():.2f}")
print(f"  Date Range: {transactions_df['order_date'].min()} to {transactions_df['order_date'].max()}")

In [ ]:
# Preview the data
print("\n👀 Customer Data Preview:")
display(customers_df.head())

print("\n👀 Transaction Data Preview:")
display(transactions_df.head())

---

## Step 3: RFM Calculation

Calculate Recency, Frequency, and Monetary values for each customer.

In [ ]:
print("🔢 Calculating RFM Metrics...")
print("=" * 70)

# Analysis date (day after last transaction)
analysis_date = transactions_df['order_date'].max() + timedelta(days=1)
print(f"Analysis Date: {analysis_date.date()}")

# Calculate RFM
rfm = transactions_df.groupby('customer_id').agg({
    'order_date': lambda x: (analysis_date - x.max()).days,  # Recency
    'transaction_id': 'count',                               # Frequency
    'order_value': 'sum'                                     # Monetary
}).reset_index()

rfm.columns = ['customer_id', 'recency', 'frequency', 'monetary']

print(f"\n✅ RFM calculated for {len(rfm):,} customers")
print(f"\n📊 RFM Summary Statistics:")
display(rfm.describe())

In [ ]:
# Preview RFM data
print("👀 Sample RFM Data:")
display(rfm.head(10))

---

## Step 4: RFM Scoring (1-5 Scale)

Assign scores using quintile method:
- **Recency**: Lower days = Higher score (5 = most recent)
- **Frequency**: More orders = Higher score
- **Monetary**: Higher spend = Higher score

In [ ]:
print("🎯 Assigning RFM Scores (Quintile Method)...")

# Quintile scoring
rfm['R_score'] = pd.qcut(rfm['recency'], 5, labels=[5,4,3,2,1], duplicates='drop')  # Reverse!
rfm['F_score'] = pd.qcut(rfm['frequency'].rank(method='first'), 5, labels=[1,2,3,4,5], duplicates='drop')
rfm['M_score'] = pd.qcut(rfm['monetary'], 5, labels=[1,2,3,4,5], duplicates='drop')

# Create RFM string (e.g., "555" for Champions)
rfm['RFM_Score'] = (rfm['R_score'].astype(str) + 
                    rfm['F_score'].astype(str) + 
                    rfm['M_score'].astype(str))

print(f"✅ Scores assigned to {len(rfm):,} customers")
print(f"\n📊 Score Distribution:")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, (score, title) in enumerate([('R_score', 'Recency'), ('F_score', 'Frequency'), ('M_score', 'Monetary')]):
    score_dist = rfm[score].value_counts().sort_index()
    axes[idx].bar(score_dist.index.astype(str), score_dist.values, color='steelblue', alpha=0.7)
    axes[idx].set_title(f'{title} Score', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Score (1=Low, 5=High)', fontsize=10)
    axes[idx].set_ylabel('Customer Count', fontsize=10)
    axes[idx].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Sample
display(rfm[['customer_id', 'recency', 'frequency', 'monetary', 'R_score', 'F_score', 'M_score', 'RFM_Score']].head(10))

---

## Step 5: Customer Segmentation

Assign customers to behavioral segments based on RFM scores.

In [ ]:
def assign_rfm_segment(row):
    """Assign segment based on RFM scores"""
    r, f, m = int(row['R_score']), int(row['F_score']), int(row['M_score'])
    
    if r >= 4 and f >= 4 and m >= 4:
        return 'Champions'
    elif r >= 3 and f >= 3 and m >= 3:
        return 'Loyal Customers'
    elif r >= 3 and f >= 2 and m >= 2:
        return 'Potential Loyalists'
    elif r >= 4 and f <= 2:
        return 'New Customers'
    elif r >= 3 and f <= 2:
        return 'Promising'
    elif r == 3 and f >= 2:
        return 'Need Attention'
    elif r == 2:
        return 'About to Sleep'
    elif r == 1 and f >= 4:
        return 'At Risk'
    elif r == 1 and m >= 4:
        return "Can't Lose Them"
    elif r == 1 and f == 2:
        return 'Hibernating'
    else:
        return 'Lost'

# Assign segments
rfm['segment'] = rfm.apply(assign_rfm_segment, axis=1)

print("🎭 Customer Segmentation Complete!")
print("\n📊 Segment Distribution:")
print(rfm['segment'].value_counts())

---

## Step 6: Segment Analysis

Analyze each segment's contribution to revenue and customer base.

In [ ]:
# Calculate segment summary
segment_summary = rfm.groupby('segment').agg({
    'customer_id': 'count',
    'recency': 'mean',
    'frequency': 'mean',
    'monetary': 'sum'
}).reset_index()

segment_summary.columns = ['Segment', 'Count', 'Avg_Recency', 'Avg_Frequency', 'Total_Revenue']
segment_summary['Pct_Customers'] = (segment_summary['Count'] / len(rfm) * 100).round(1)
segment_summary['Pct_Revenue'] = (segment_summary['Total_Revenue'] / rfm['monetary'].sum() * 100).round(1)
segment_summary['Avg_Customer_Value'] = (segment_summary['Total_Revenue'] / segment_summary['Count']).round(2)
segment_summary = segment_summary.sort_values('Total_Revenue', ascending=False)

print("\n📊 SEGMENT PERFORMANCE SUMMARY")
print("=" * 100)
display(segment_summary.style.background_gradient(subset=['Total_Revenue'], cmap='Greens'))

---

## Step 7: Key Business Insights

In [ ]:
print("🔍 KEY BUSINESS INSIGHTS")
print("=" * 70)

total_revenue = rfm['monetary'].sum()

# Top segments
print("\n💎 Top 3 Revenue-Generating Segments:")
top_3 = segment_summary.head(3)
for idx, row in top_3.iterrows():
    print(f"  {row['Segment']}: {int(row['Count']):,} customers ({row['Pct_Customers']}%) = ₹{row['Total_Revenue']/1e7:.2f} Cr ({row['Pct_Revenue']}%)")

# Champions analysis
champions = rfm[rfm['segment'] == 'Champions']
print(f"\n👑 Champions Deep Dive:")
print(f"  Count: {len(champions):,} customers ({len(champions)/len(rfm)*100:.1f}% of base)")
print(f"  Revenue: ₹{champions['monetary'].sum()/1e7:.2f} Cr ({champions['monetary'].sum()/total_revenue*100:.1f}% of total)")
print(f"  Avg Spend: ₹{champions['monetary'].mean():,.2f}")
print(f"  Avg Orders: {champions['frequency'].mean():.1f}")
print(f"  Avg Recency: {champions['recency'].mean():.0f} days")

# At-risk
at_risk = rfm[rfm['segment'].isin(['At Risk', "Can't Lose Them"])]
print(f"\n⚠️  At-Risk Customers:")
print(f"  Count: {len(at_risk):,} customers")
print(f"  Historical Value: ₹{at_risk['monetary'].sum()/1e7:.2f} Cr (at risk of being lost)")
print(f"  Recovery Potential (30-40%): ₹{at_risk['monetary'].sum()*0.35/1e7:.1f} Cr")

# New customers
new_cust = rfm[rfm['segment'] == 'New Customers']
print(f"\n🆕 New Customers:")
print(f"  Count: {len(new_cust):,} customers")
print(f"  Avg Orders: {new_cust['frequency'].mean():.1f} (conversion opportunity!)")
print(f"  Potential if converted to Loyal: 3-4x LTV increase")

---

## Step 8: Visualizations

In [ ]:
# 1. Segment Distribution (Pie Chart)
fig, ax = plt.subplots(figsize=(10, 7))
segment_counts = rfm['segment'].value_counts()
colors = plt.cm.Set3(range(len(segment_counts)))
wedges, texts, autotexts = ax.pie(
    segment_counts.values, 
    labels=segment_counts.index,
    autopct='%1.1f%%',
    colors=colors,
    startangle=90
)
ax.set_title('Customer Segmentation Distribution', fontsize=16, fontweight='bold', pad=20)
for autotext in autotexts:
    autotext.set_color('black')
    autotext.set_fontweight('bold')
plt.tight_layout()
plt.show()

In [ ]:
# 2. Revenue by Segment (Bar Chart)
fig, ax = plt.subplots(figsize=(12, 6))
segment_summary_sorted = segment_summary.sort_values('Total_Revenue', ascending=True)
bars = ax.barh(segment_summary_sorted['Segment'], segment_summary_sorted['Total_Revenue']/1e7, color='steelblue')
ax.set_xlabel('Total Revenue (₹ Crore)', fontsize=12)
ax.set_title('Revenue Contribution by Segment', fontsize=16, fontweight='bold', pad=20)
ax.grid(axis='x', alpha=0.3)

# Add value labels
for i, (segment, revenue) in enumerate(zip(segment_summary_sorted['Segment'], segment_summary_sorted['Total_Revenue'])):
    ax.text(revenue/1e7 + 0.5, i, f'₹{revenue/1e7:.1f}Cr', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# 3. RFM Scatter Plot
fig, ax = plt.subplots(figsize=(12, 8))
scatter = ax.scatter(
    rfm['recency'], 
    rfm['monetary'],
    c=rfm['frequency'],
    s=50,
    alpha=0.6,
    cmap='viridis'
)
ax.set_xlabel('Recency (days)', fontsize=12)
ax.set_ylabel('Monetary Value (₹)', fontsize=12)
ax.set_title('Customer Distribution: Recency vs Monetary (Color = Frequency)', fontsize=16, fontweight='bold', pad=20)
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Frequency (# orders)', fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 4. Segment Metrics Comparison
segment_metrics = rfm.groupby('segment').agg({
    'recency': 'mean',
    'frequency': 'mean',
    'monetary': 'mean'
}).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Avg Recency
axes[0].barh(segment_metrics['segment'], segment_metrics['recency'], color='coral')
axes[0].set_xlabel('Days Since Last Order')
axes[0].set_title('Average Recency by Segment', fontweight='bold')
axes[0].grid(axis='x', alpha=0.3)

# Avg Frequency
axes[1].barh(segment_metrics['segment'], segment_metrics['frequency'], color='lightgreen')
axes[1].set_xlabel('Number of Orders')
axes[1].set_title('Average Frequency by Segment', fontweight='bold')
axes[1].grid(axis='x', alpha=0.3)

# Avg Monetary
axes[2].barh(segment_metrics['segment'], segment_metrics['monetary'], color='skyblue')
axes[2].set_xlabel('Average Spend (₹)')
axes[2].set_title('Average Monetary by Segment', fontweight='bold')
axes[2].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

---

## Step 9: Campaign Recommendations

Data-driven marketing strategies for each segment.

In [ ]:
campaigns = {
    'Champions': {
        'Campaign': 'Levi\'s Black VIP Program',
        'Offer': 'Early access to new collections, free alterations, 25% birthday discount',
        'Goal': 'Retain and maximize LTV',
        'Expected_ROI': '6-8x'
    },
    'Loyal Customers': {
        'Campaign': 'Referral Rewards',
        'Offer': 'Refer a friend, both get ₹500 off',
        'Goal': 'Leverage for acquisition',
        'Expected_ROI': '4-5x'
    },
    'New Customers': {
        'Campaign': 'Complete Your Look',
        'Offer': '15% off 2nd purchase within 30 days',
        'Goal': 'Drive second purchase (conversion)',
        'Expected_ROI': '5-7x'
    },
    'At Risk': {
        'Campaign': 'Win-Back Special',
        'Offer': '30% off + free shipping (7-day urgency)',
        'Goal': 'Reactivate before churn',
        'Expected_ROI': '7-9x'
    },
    "Can't Lose Them": {
        'Campaign': 'Personal Outreach',
        'Offer': 'Call from relationship manager + exclusive 40% off',
        'Goal': 'Save high-value customers',
        'Expected_ROI': '10-12x'
    }
}

print("🎯 RECOMMENDED MARKETING CAMPAIGNS")
print("=" * 70)

campaign_data = []
for segment, details in campaigns.items():
    customer_count = len(rfm[rfm['segment'] == segment])
    print(f"\n📌 {segment} ({customer_count:,} customers)")
    print(f"   Campaign: {details['Campaign']}")
    print(f"   Offer: {details['Offer']}")
    print(f"   Goal: {details['Goal']}")
    print(f"   Expected ROI: {details['Expected_ROI']}")
    
    campaign_data.append({
        'Segment': segment,
        'Customers': customer_count,
        'Campaign': details['Campaign'],
        'Expected_ROI': details['Expected_ROI']
    })

# Create campaign summary table
campaign_df = pd.DataFrame(campaign_data)
print("\n\n📊 Campaign Summary Table:")
display(campaign_df)

---

## Step 10: Executive Summary

In [ ]:
print("\n" + "=" * 70)
print("📈 EXECUTIVE SUMMARY")
print("=" * 70)

total_revenue = rfm['monetary'].sum()
top_3_segments_revenue = segment_summary.head(3)['Total_Revenue'].sum()
top_3_segments_count = segment_summary.head(3)['Count'].sum()

print(f"""
Key Findings:
• Total Customer Base: {len(rfm):,} customers
• Total Revenue Analyzed: ₹{total_revenue/1e7:.2f} Crore
• Top 3 Segments: {int(top_3_segments_count):,} customers ({top_3_segments_count/len(rfm)*100:.1f}%)
  → Generate: ₹{top_3_segments_revenue/1e7:.2f} Cr ({top_3_segments_revenue/total_revenue*100:.1f}% of revenue)

Champion Segment:
• {len(champions):,} customers ({len(champions)/len(rfm)*100:.1f}% of base)
• ₹{champions['monetary'].sum()/1e7:.2f} Cr revenue ({champions['monetary'].sum()/total_revenue*100:.1f}% of total)
• 10x more valuable than average customer

At-Risk Opportunity:
• {len(at_risk):,} high-value customers at risk
• ₹{at_risk['monetary'].sum()/1e7:.2f} Cr historical value to protect
• Win-back campaign could recover 30-40% → ₹{at_risk['monetary'].sum()*0.35/1e7:.1f}Cr potential

Recommended Actions:
1. Launch VIP program for Champions (protect ₹{champions['monetary'].sum()/1e7:.1f}Cr segment)
2. 2nd-purchase incentive for {len(new_cust):,} New Customers (conversion opportunity)
3. Urgent win-back for {len(at_risk):,} At-Risk customers (₹{at_risk['monetary'].sum()/1e7:.1f}Cr at stake)
4. Suppress marketing to Lost segment (save ₹2-3Cr annually)
""")

print("=" * 70)
print("✅ ANALYSIS COMPLETE!")
print("=" * 70)

---

## Step 11: Export Results

In [ ]:
# Save datasets
customers_df.to_csv('levis_customers.csv', index=False)
transactions_df.to_csv('levis_transactions.csv', index=False)
rfm.to_csv('levis_rfm_analysis.csv', index=False)
segment_summary.to_csv('levis_segment_summary.csv', index=False)

print("💾 Files saved:")
print("  ✓ levis_customers.csv")
print("  ✓ levis_transactions.csv")
print("  ✓ levis_rfm_analysis.csv")
print("  ✓ levis_segment_summary.csv")

---

## 🎓 Key Learnings

### 1. Pareto Principle in Action
**16.8% of customers (Champions) = 33% of revenue**. Focus retention here for maximum ROI.

### 2. Early Intervention > Recovery
Converting **New Customers** with 1 order is easier than winning back **Lost** customers.

### 3. Segment-Specific Strategies
- **Champions** → Exclusivity & VIP treatment
- **At-Risk** → Urgency & discounts
- **New** → Onboarding & second purchase

### 4. Resource Allocation
Suppress marketing to **Lost** segment (9.7% of base, <4% of revenue) → Save ₹2-3 Cr annually.

---

## 📌 Next Steps

1. **CLV Modeling**: Predict lifetime value using BG/NBD model
2. **Churn Prediction**: Build ML model (XGBoost) to predict churn 30-60 days in advance
3. **Cohort Analysis**: Track retention curves by acquisition cohort
4. **A/B Testing**: Test campaign effectiveness experimentally

---

**Project by**: [Your Name]  
**GitHub**: [Your GitHub]  
**LinkedIn**: [Your LinkedIn]